# CropCheckUp — leaf disease classifier

Fine-tunes an ImageNet-pretrained ConvNeXt-Tiny (timm) on the BiRefNet-segmented PlantVillage images
(`plantvillage/segmented/<class>/*.png`, 38 classes) and exports it to ONNX for the CropCheckUp app.

- **Input contract:** each RGBA PNG is composited onto black and resized to 224×224, the same way the app prepares a
  photo after background removal. The exported model takes `image` as float32 `[N, 224, 224, 3]` RGB in 0–255,
  does its own normalisation and returns softmax `probabilities` `[N, 38]` in `labels.txt` order.
- **Split:** 80/10/10 by a hash of each photo's ID, so PlantVillage's duplicates (`<id> copy 2`, `<id>.1`,
  `<id> 1`, the same leaf on different days) never straddle train and test.
- **Augmentation:** rotation, flips, a tight random-resized crop (down to 25% of the image, so black borders and
  framing aren't a reliable cue: Corn healthy is the only class shot with no background), colour jitter and blur.
- **Imbalance:** classes range from 152 to 5,507 images. Training samples each class in proportion to √count,
  halfway between natural and fully balanced, plus label smoothing.
- **Evaluation:** test accuracy and balanced accuracy, per-class recall, the most frequent confusions, accuracy on
  the images BiRefNet kept whole, and a centre-crop pass that hides the background, to check for a border shortcut.

Output in `/kaggle/working`: `plant_disease_model.onnx`, `labels.txt`, `metrics.json`, `history.json`,
`split.json`, `best.pt`.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "onnx", "onnxscript", "onnxruntime"])

In [ ]:
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

MODEL_NAME = "convnext_tiny.fb_in22k_ft_in1k"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
SAMPLER_POWER = 0.5  # class c is drawn in proportion to count**SAMPLER_POWER: 1 is natural, 0 fully balanced
MIN_CROP = 0.25  # smallest random-resized-crop area, as a fraction of the image
CENTRE_CROP = 0.6  # side of the background-hiding test crop, as a fraction of the image
SPLIT = (0.8, 0.1, 0.1)
NUM_WORKERS = 4
MAX_PER_CLASS = None  # set to e.g. 50 for a quick end-to-end test run
SEED = 0

SEG_DIR = next(p for p in Path("/kaggle/input").rglob("segmented") if p.parent.name == "plantvillage")
WORK_DIR = Path("/kaggle/working")

In [ ]:
def group_key(path):
    # Copies of one photo or leaf share the ID up to its first number: "RS_HL 5698 1", "R.S_HL 8129 copy 2",
    # "GH_HL Leaf 238.1". A few Corn rust files have no "<uuid>___" prefix.
    stem = path.stem.split("___", 1)[-1]
    return f"{path.parent.name}/{re.match(r'^\D*\d+', stem).group(0)}"


def split_of(path):
    h = int(hashlib.md5(group_key(path).encode()).hexdigest(), 16) % 10_000 / 10_000
    return "train" if h < SPLIT[0] else "val" if h < SPLIT[0] + SPLIT[1] else "test"


manifest = json.loads((SEG_DIR / "manifest.json").read_text())
removed = set(manifest.get("removed", {}))
kept_whole = set(manifest["fallback_full_image"])

classes = sorted(d.name for d in SEG_DIR.iterdir() if d.is_dir())
items = {"train": [], "val": [], "test": []}
for label, name in enumerate(classes):
    paths = sorted(p for p in (SEG_DIR / name).glob("*.png") if f"{name}/{p.name}" not in removed)
    for p in paths[:MAX_PER_CLASS]:
        items[split_of(p)].append((p, label))

print(f"{len(classes)} classes, {sum(map(len, items.values()))} images (manifest total {manifest['total']})")
for split, rows in items.items():
    counts = Counter(label for _, label in rows)
    print(f"{split:5} {len(rows):6}  smallest class {min(counts.values())}, largest {max(counts.values())}")

(WORK_DIR / "labels.txt").write_text("\n".join(classes) + "\n")
(WORK_DIR / "split.json").write_text(json.dumps(
    {split: [f"{p.parent.name}/{p.name}" for p, _ in rows] for split, rows in items.items()}, indent=1))

In [ ]:
import math
import time

import numpy as np
import timm
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from timm.data import resolve_data_config
from timm.optim import create_optimizer_v2
from torchvision import transforms as T

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"  # T4 has no bf16, so fp16 with a grad scaler

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(classes), drop_path_rate=0.1)
data_cfg = resolve_data_config({}, model=model)
MEAN, STD = data_cfg["mean"], data_cfg["std"]
model = model.to(device, memory_format=torch.channels_last)

BILINEAR = T.InterpolationMode.BILINEAR  # the app resizes with PIL bilinear
train_tf = T.Compose([
    T.RandomRotation(180, interpolation=BILINEAR, fill=0),
    T.RandomResizedCrop(IMG_SIZE, scale=(MIN_CROP, 1.0), interpolation=BILINEAR),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), interpolation=BILINEAR), T.ToTensor(), T.Normalize(MEAN, STD)])
centre_tf = T.Compose([
    T.Lambda(lambda im: T.functional.center_crop(im, [round(CENTRE_CROP * s) for s in im.size[::-1]])),
    eval_tf,
])


def load_rgb(path):
    # Same as the app: composite the cut-out leaf onto black.
    with Image.open(path) as im:
        rgba = im.convert("RGBA")
    return Image.alpha_composite(Image.new("RGBA", rgba.size, (0, 0, 0, 255)), rgba).convert("RGB")


class LeafDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        path, label = self.rows[i]
        return self.transform(load_rgb(path)), label


def loader(rows, transform, sampler=None):
    return DataLoader(LeafDataset(rows, transform), batch_size=BATCH_SIZE, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=use_amp, persistent_workers=NUM_WORKERS > 0)


train_counts = Counter(label for _, label in items["train"])
sample_weights = [train_counts[label] ** (SAMPLER_POWER - 1) for _, label in items["train"]]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(items["train"]), replacement=True)
train_loader = loader(items["train"], train_tf, sampler)
val_loader = loader(items["val"], eval_tf)

In [ ]:
@torch.no_grad()
def predict(model, data):
    model.eval()
    probs, labels = [], []
    for x, y in data:
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            out = model(x.to(device, non_blocking=True).to(memory_format=torch.channels_last))
        probs.append(out.float().softmax(-1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def scores(probs, labels):
    pred = probs.argmax(1)
    recall = {classes[c]: float((pred[labels == c] == c).mean()) for c in np.unique(labels)}
    return {"accuracy": float((pred == labels).mean()), "balanced_accuracy": float(np.mean(list(recall.values()))),
            "recall": recall}


optimizer = create_optimizer_v2(model, "adamw", lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
warmup_steps = len(train_loader)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min(1.0, (s + 1) / warmup_steps)
                                              * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps))))
scaler = torch.amp.GradScaler(enabled=use_amp)
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

history, best = [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    start, running, seen = time.time(), 0.0, 0
    for x, y in train_loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running, seen = running + loss.item() * len(y), seen + len(y)

    val = scores(*predict(model, val_loader))
    row = {"epoch": epoch, "train_loss": running / seen, "val_accuracy": val["accuracy"],
           "val_balanced_accuracy": val["balanced_accuracy"], "seconds": round(time.time() - start)}
    history.append(row)
    print(row)
    if val["balanced_accuracy"] > best:
        best = val["balanced_accuracy"]
        torch.save(model.state_dict(), WORK_DIR / "best.pt")

(WORK_DIR / "history.json").write_text(json.dumps(history, indent=1))

In [ ]:
model.load_state_dict(torch.load(WORK_DIR / "best.pt", map_location=device))
probs, labels = predict(model, loader(items["test"], eval_tf))
centre_probs, _ = predict(model, loader(items["test"], centre_tf))
pred = probs.argmax(1)

test_names = [f"{p.parent.name}/{p.name}" for p, _ in items["test"]]
whole = np.array([name in kept_whole for name in test_names])
confusions = Counter((classes[t], classes[p]) for t, p in zip(labels, pred) if t != p)

metrics = {
    "model": MODEL_NAME,
    "best_val_balanced_accuracy": best,
    "test": scores(probs, labels),
    "test_centre_crop": scores(centre_probs, labels),
    "test_kept_whole": {"images": int(whole.sum()), "accuracy": float((pred[whole] == labels[whole]).mean())},
    "top_confusions": [{"true": t, "predicted": p, "count": n} for (t, p), n in confusions.most_common(20)],
}
(WORK_DIR / "metrics.json").write_text(json.dumps(metrics, indent=1))

print(f"test accuracy {metrics['test']['accuracy']:.4f}, balanced {metrics['test']['balanced_accuracy']:.4f}")
print(f"centre crop   {metrics['test_centre_crop']['accuracy']:.4f}, "
      f"balanced {metrics['test_centre_crop']['balanced_accuracy']:.4f}")
print(f"kept whole    {metrics['test_kept_whole']['accuracy']:.4f} on {metrics['test_kept_whole']['images']} images")
print("\nlowest recall (full / centre crop):")
for name, r in sorted(metrics["test"]["recall"].items(), key=lambda kv: kv[1])[:8]:
    print(f"  {r:.3f} / {metrics['test_centre_crop']['recall'][name]:.3f}  {name}")
print("\ntop confusions:")
for c in metrics["top_confusions"][:10]:
    print(f"  {c['count']:3}  {c['true']} -> {c['predicted']}")

In [ ]:
import onnxruntime as ort


class AppModel(nn.Module):
    """Takes the app's input, float32 [N, 224, 224, 3] RGB in 0-255, and returns softmax probabilities."""

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(MEAN).view(1, 3, 1, 1) * 255)
        self.register_buffer("std", torch.tensor(STD).view(1, 3, 1, 1) * 255)

    def forward(self, image):
        return self.model((image.permute(0, 3, 1, 2) - self.mean) / self.std).softmax(-1)


app_model = AppModel(model.float().cpu().to(memory_format=torch.contiguous_format)).eval()
onnx_path = WORK_DIR / "plant_disease_model.onnx"
# The dummy batch is 2: a batch of 1 would be specialised to a constant and clash with the dynamic dim.
torch.onnx.export(app_model, (torch.zeros(2, IMG_SIZE, IMG_SIZE, 3),), onnx_path,
                  input_names=["image"], output_names=["probabilities"],
                  dynamic_shapes=({0: torch.export.Dim("batch", max=1024)},), external_data=False)

# Check the ONNX model against PyTorch on real test images, fed the way the app feeds them.
batch = np.stack([np.asarray(load_rgb(p).resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR), np.float32)
                  for p, _ in items["test"][:16]])
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_probs = session.run(None, {"image": batch})[0]
with torch.no_grad():
    torch_probs = app_model(torch.from_numpy(batch)).numpy()
print(f"ONNX {onnx_path.stat().st_size / 1e6:.1f} MB, max |onnx - torch| = {np.abs(onnx_probs - torch_probs).max():.2e}, "
      f"argmax agrees on {(onnx_probs.argmax(1) == torch_probs.argmax(1)).mean():.0%}")